# Lab 06-03 · Kirchhoff's Laws, Dividers, Bridges and Delta–Wye

Companion to chapter [06-03](../../chapters/06-03-kirchhoffs-laws.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None

def solve_resistive(nodes, resistors, current_in):
    '''Node voltages of a resistor network (ground = "0") with currents injected at nodes.
    resistors: list of (node_a, node_b, ohms); current_in: dict node -> amperes injected.'''
    idx = {n: k for k, n in enumerate(nodes)}
    G = np.zeros((len(nodes), len(nodes)))
    for a, b, r in resistors:
        for p, q in ((a, b), (b, a)):
            if p != "0":
                G[idx[p], idx[p]] += 1 / r
                if q != "0":
                    G[idx[p], idx[q]] -= 1 / r
    i = np.array([current_in.get(n, 0.0) for n in nodes])
    v = np.linalg.solve(G, i)
    return {"0": 0.0, **dict(zip(nodes, v))}

print("ngspice available:", HAVE_NGSPICE)

ngspice available: True


## Part 1 — Kirchhoff by simulation

Example 06-03.5: 10 V between node a and ground; $R_{ac}$ = 10, $R_{ad}$ = 20, $R_{cd}$ = 30, $R_{cb}$ = 40, $R_{db}$ = 50 Ω.

**🔮 Predict:** with 4 nodes (a, c, d, ground) and 6 branches (five resistors and the source), how many independent KCL and KVL equations?

In [2]:
res = [("a", "c", 10.0), ("a", "d", 20.0), ("c", "d", 30.0), ("c", "0", 40.0), ("d", "0", 50.0)]
# find the source current that makes v_a = 10 V: solve with 1 A injected, then scale (linearity)
v1 = solve_resistive(["a", "c", "d"], res, {"a": 1.0})
R_eq = v1["a"]
v = {k: x * 10 / R_eq for k, x in v1.items()}
I_src = 10 / R_eq
print(f"R_eq = {R_eq:.3f} ohm, source current {I_src:.4f} A, v_c = {v['c']:.3f} V, v_d = {v['d']:.3f} V")
n_nodes, n_branches = 4, 6
print(f"independent KCL: {n_nodes - 1}, independent KVL: {n_branches - n_nodes + 1}")

def i_res(a, b, r):                # current from a to b through resistor r
    return (v[a] - v[b]) / r

kcl = {
    "a": I_src - i_res("a", "c", 10) - i_res("a", "d", 20),
    "c": i_res("a", "c", 10) - i_res("c", "d", 30) - i_res("c", "0", 40),
    "d": i_res("a", "d", 20) + i_res("c", "d", 30) - i_res("d", "0", 50),
    "ground": i_res("c", "0", 40) + i_res("d", "0", 50) - I_src,
}
kvl = {   # meshes: a-c-d-a, c-0-d-c, and source-a-d-0
    "a-c-d": (v["a"] - v["c"]) + (v["c"] - v["d"]) + (v["d"] - v["a"]),
    "c-0-d": (v["c"] - 0) + (0 - v["d"]) + (v["d"] - v["c"]),
    "src-a-d-0": -10 + (v["a"] - v["d"]) + v["d"],
}
print("KCL residuals (A):", {k: f"{x:.1e}" for k, x in kcl.items()})
print("KVL residuals (V):", {k: f"{x:.1e}" for k, x in kvl.items()})
assert max(abs(x) for x in kcl.values()) < 1e-12 and max(abs(x) for x in kvl.values()) < 1e-12
print(f"bridge current c->d: {i_res('c', 'd', 30) * 1e3:.2f} mA")

R_eq = 29.048 ohm, source current 0.3443 A, v_c = 7.869 V, v_d = 7.377 V
independent KCL: 3, independent KVL: 3
KCL residuals (A): {'a': '2.8e-17', 'c': '0.0e+00', 'd': '1.4e-16', 'ground': '-1.7e-16'}
KVL residuals (V): {'a-c-d': '0.0e+00', 'c-0-d': '0.0e+00', 'src-a-d-0': '0.0e+00'}
bridge current c->d: 16.39 mA


In [3]:
if HAVE_NGSPICE:
    out = subprocess.run(["ngspice", "-b", "networks.cir"], capture_output=True, text=True).stdout
    for line in out.splitlines():
        if " = " in line and line.split("=")[0].strip() in ("ex5i", "ex5icd", "ex2i", "ex2v4"):
            print("  ngspice", line.strip())
else:
    print("ngspice not found; see SETUP.md. Skipping.")

  ngspice ex2i = -3
  ngspice ex2v4 = 8
  ngspice ex5i = -0.344262
  ngspice ex5icd = 0.0163934


The KVL around the first two meshes is satisfied trivially once voltages are node-voltage differences: that is the sense in which node voltages "build in" KVL, the idea nodal analysis (06-04) exploits. The KVL equation that carries information is the one through the source.

## Part 2 — Loading a divider

**🔮 Predict:** Example 06-03.3's divider (100 kΩ over 33 kΩ, source resistance 24.8 kΩ) feeds a load $R_L$. What $R_L$ keeps the error below 0.1%?

In [4]:
R1, R2 = 100e3, 33e3
Rs = R1 * R2 / (R1 + R2)
RL = np.logspace(4, 8, 401)
err = Rs / (Rs + RL)                       # fractional error of the loaded output
RL_01 = Rs * (1 / 1e-3 - 1)
print(f"source resistance {Rs / 1e3:.2f} kohm; 0.1% error needs R_L >= {RL_01 / 1e6:.1f} Mohm")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.loglog(RL, err * 100)
ax.axhline(0.1, color="0.5", ls="--", lw=0.8)
ax.set_xlabel("load resistance (ohm)"); ax.set_ylabel("output error (%)"); ax.grid(True, which="both", alpha=0.3)
fig.tight_layout(); plt.show()

source resistance 24.81 kohm; 0.1% error needs R_L >= 24.8 Mohm


The error is $R_s/(R_s + R_L)$: 0.1% needs a load 999 times the source resistance, about 25 MΩ. Typical meters (10 MΩ) and many ADC inputs fall short of that, which is why precision dividers are buffered or kept low in resistance.

## Part 3 — Bridge linearity

**🔮 Predict:** which of the quarter bridge (one active gauge), half bridge (two gauges, equal and opposite) and full bridge (four gauges) is exactly linear?

In [5]:
V, GF = 2.5, 2.0
eps = np.linspace(-5000e-6, 5000e-6, 201)
x = GF * eps
quarter = V * ((1 + x) / (2 + x) - 0.5)
half = V * ((1 + x) / 2 - 0.5)                      # R(1+x) and R(1-x) in one divider
full = V * ((1 + x) / 2 - (1 - x) / 2)               # both dividers active, opposite
for name, out, ideal in (("quarter", quarter, V * x / 4), ("half", half, V * x / 2), ("full", full, V * x)):
    dev = (out[-1] / ideal[-1] - 1) * 100
    print(f"{name:7} bridge: output at +5000 ue = {out[-1] * 1e3:7.3f} mV, deviation from linear {dev:+.2f}%")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(eps * 1e6, quarter * 1e3, label="quarter"); ax.plot(eps * 1e6, half * 1e3, label="half"); ax.plot(eps * 1e6, full * 1e3, label="full")
ax.set_xlabel("strain (µε)"); ax.set_ylabel("bridge output (mV)"); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

quarter bridge: output at +5000 ue =   6.219 mV, deviation from linear -0.50%
half    bridge: output at +5000 ue =  12.500 mV, deviation from linear +0.00%
full    bridge: output at +5000 ue =  25.000 mV, deviation from linear +0.00%


The quarter bridge's output bends (−0.50% at +5000 µε), because the active arm changes its divider's total resistance. In the half and full bridges each divider's total stays $2R$, so the output is exactly proportional to strain, and two or four times larger.

## Part 4 — Delta–wye by numbers

In [6]:
rng = np.random.default_rng(3)
def d2y(rab, rbc, rca):
    s = rab + rbc + rca
    return rab * rca / s, rab * rbc / s, rbc * rca / s          # Ra, Rb, Rc
def y2d(ra, rb, rc):
    p = ra * rb + rb * rc + rc * ra
    return p / rc, p / ra, p / rb                                # Rab, Rbc, Rca
worst = 0.0
for _ in range(1000):
    delta = rng.uniform(1, 1000, 3)
    back = np.array(y2d(*d2y(*delta)))
    worst = max(worst, np.max(np.abs(back / delta - 1)))
    ra, rb, rc = d2y(*delta)
    rab_delta = delta[0] * (delta[1] + delta[2]) / delta.sum()   # a-b with c open, delta
    assert abs((ra + rb) / rab_delta - 1) < 1e-12
print(f"largest relative round-trip error over 1000 random deltas: {worst:.1e}")

largest relative round-trip error over 1000 random deltas: 4.4e-16


## Your turn

1. In Part 1, change $R_{db}$ so that the bridge balances. What value, and what is the bridge current then?
2. In Part 2, a buffer amplifier with 1 TΩ input resistance replaces the ADC. What is the error?
3. In Part 3, what quarter-bridge output does 5000 µε give at 5 V excitation, and what happens to the gauge's self-heating?

<details><summary>Answers</summary>

1. Balance needs $R_{ac}R_{db} = R_{ad}R_{cb}$: $10 R_{db} = 20\times40$, so $R_{db}$ = 80 Ω. The bridge current is then zero, and the 30 Ω resistor can be removed without changing anything else: the network becomes two series pairs in parallel, $(10 + 40)\parallel(20 + 80) = 33.3$ Ω.
2. $24.8\ \text{kΩ}/10^{12}\ \Omega \approx 2.5\times10^{-8}$, so 2.5 ppm (0.0000025%): negligible; the buffer's offset voltage now dominates the error (06-10).
3. The output doubles with excitation: $5\times(1.01/2.01 - 0.5) = 12.44$ mV (the linear estimate is 12.5 mV). Each arm now has 2.5 V across it, so a 350 Ω gauge dissipates $2.5^2/350 = 17.9$ mW, four times as much; the extra self-heating warms the gauge and can cause drift, which is why excitation is limited.

</details>